# Explore dataset structure

In [1]:
import os
from pathlib import Path
os.chdir(Path.cwd().parent)   # go one level up
print(os.getcwd())         

from xflow.utils.io import scan_files
from xflow.utils.sql import merge_sqlite_dbs

from config_utils import load_config, detect_machine
import pandas as pd
from utils import *

experiment_name = "CLEAR_visualization"  
machine = detect_machine() 
config = load_config(
    f"{experiment_name}.yaml",
    machine=machine
)

# if win, windows test in machine then auto switch to windows path, otherwise use mac path
if any(win in machine for win in ["win", "windows"]): # windows
    dirs = {
        "Dataset_root_dir": "C:/Users/qiyuanxu/Documents/DataHub/processed/",
        "DMD_only": {
            "dataset_db_dir": "C:/Users/qiyuanxu/Documents/DataHub/processed/CLEAR25_DMD/db/dataset_meta.db",
            "dataset_extracted_dir": "C:/Users/qiyuanxu/Documents/DataHub/processed/CLEAR25_DMD/",
        },
        "Chromox_only": {
            "dataset_db_dir": "C:/Users/qiyuanxu/Documents/DataHub/processed/CLEAR25_Chromox/db/dataset_meta.db",
            "dataset_extracted_dir": "C:/Users/qiyuanxu/Documents/DataHub/processed/CLEAR25_Chromox/",
        }
    }

elif any(mac in machine for mac in ["mac", "darwin"]): # Mac
    dirs = {
        "DMD_only": {
            "dataset_db_dir": "",
        },
        "Chromox_only": {
            "dataset_db_dir": "",
        }
    }

else:
    raise ValueError(f"Unsupported machine: {machine}")

c:\Users\qiyuanxu\Documents\GitHub\fiber-image-reconstruction-comparison
[config_utils] Using machine profile: win-qiyuanxu


# Load dataset

Different with training pipeline, data sample meta data is given

In [2]:
# ============================
# Merge entire CLEAR 2025 dataset in to a single database
# ============================
db_paths = scan_files(dirs["Dataset_root_dir"], extensions=[".db"], return_type="str")
conn = merge_sqlite_dbs(db_paths, source_column="db_path")
sql = """
SELECT *
FROM mmf_dataset_metadata
"""
tables_df = pd.read_sql_query(sql, conn)
conn.close()
# optional: drop duplicate column names (e.g. both tables have "id", "db_path")
tables_df = tables_df.loc[:, ~tables_df.columns.duplicated()]
print(tables_df.shape)

(3625, 20)


In [3]:
# pip install xflow-py
from xflow import SqlProvider, PyTorchPipeline
from xflow.data import build_transforms_from_config
from xflow.extensions.physics.pipeline import CachedBasisPipeline, IndexCombinator
from xflow.extensions.physics import pattern_gen

from tqdm import tqdm

dmd_orth_basis_provider = SqlProvider(
    sources={"connection": dirs["DMD_only"]["dataset_db_dir"], "sql": config["sql"]["train"]}, output_config={'list': "image_path"}
)

dmd_validation_provider = SqlProvider(
    sources={"connection": dirs["DMD_only"]["dataset_db_dir"], "sql": config["sql"]["eval"]}, output_config={'list': "image_path"}
)

config["data"]["transforms"]["torch"].insert(0, {
    "name": "add_parent_dir",
    "params": {
        "parent_dir": dirs["DMD_only"]["dataset_extracted_dir"]
    }
})
transforms = build_transforms_from_config(config["data"]["transforms"]["torch"])

val_pipeline = PyTorchPipeline(
    dmd_validation_provider, 
    transforms[:-1],
).to_numpy() 

# ========== SGM simulation pattern generator ==========
canvas = pattern_gen.DynamicPatterns(*config["simulation"]["canvas_size"])
canvas.set_postprocess_fns(build_transforms_from_config(config["simulation"]["process_functions"]))
canvas._distributions = [pattern_gen.StaticGaussianDistribution(canvas) for _ in range(config["simulation"]["total_Guassian_num"])]
canvas.set_threshold(config["simulation"]["minimum_pixel_threshold"])
stream = canvas.pattern_stream(std_1=config["simulation"]["std_1"], std_2=config["simulation"]["std_2"],
                               max_intensity=config["simulation"]["max_intensity"], fade_rate=config["simulation"]["fade_rate"], 
                               distribution=config["simulation"]["distribution"]) 

# ======== random combinator using index + SGM ========
combinator = IndexCombinator(
    pattern_provider=stream,
    transforms= build_transforms_from_config(config["combinator"]["transforms"]["torch"]),
)

train_dataset = CachedBasisPipeline(
    dmd_orth_basis_provider, 
    combinator=combinator, 
    transforms=transforms, 
    num_samples=config["data"]["total_train_samples"], 
    seed=config["seed"],
    eager=True
)

samples = []
for _ in tqdm(range(1000)):
    samples.append(next(iter(train_dataset)))
samples = np.stack(samples, axis=0)         # (1000, 2, 1, 256, 256)
samples = np.transpose(samples, (1, 0, 2, 3, 4))  # (2, 1000, 1, 256, 256)

100%|██████████| 1000/1000 [00:24<00:00, 40.93it/s]               


# Latent space distribution

In [4]:
labels = ["DMD_syth (training set)"] * len(samples[0]) + ["DMD_val (test set)"] * len(val_pipeline[0])
result = np.concatenate([samples, np.stack(val_pipeline, axis=0)], axis=1) 
print(result.shape)

(2, 1067, 1, 256, 256)


In [7]:
import numpy as np
import imageio.v2 as imageio
from pathlib import Path
from xflow.utils.visualization import DimReducer, Embedding3DPlot

# 1) Prepare PCA, t-SNE, UMAP coords
method = "t-SNE"
index = 0  # 0 for fiber output speckle, 1 for original image
title = "{} projection - MMF output speckle".format(method.upper()) if index == 0 else "{} projection - Original images".format(method.upper())
X = result[index].reshape(result[index].shape[0], -1)  
coords = DimReducer(method=method, n_components=3, random_state=42).fit_transform(X) 

# 2) Configure plotter once (object-level defaults)
plotter = Embedding3DPlot(
    coords=coords,
    labels=labels,
    title=title,
    point_size=4,

    # Shared defaults for both Plotly projections and Matplotlib frames
    show_projections=False,      # set True if you want wall projection points
    projection_envelope=True,    # shading on wall distributions
    projection_alpha=0.25,
    projection_size_scale=0.7,
    projection_gap_ratio=0.06,
    projection_envelope_alpha=0.18,
)
# plotter.show(elev=25, azim=35); 

# 3) Save orbit GIF from Matplotlib frames
out_dir = Path("results/gif")
out_dir.mkdir(parents=True, exist_ok=True)

frames = []
n_frames = 120
base_elev, base_azim = 25.0, 40.0
azim_radius = 25.0 / 2.0
elev_radius = 10.0 / 2.0
t = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)

for tt in t:
    azim = base_azim + azim_radius * np.cos(tt)
    elev = base_elev + elev_radius * np.sin(tt)

    frame = plotter.get_matplotlib_frame(
        elev=float(elev),
        azim=float(azim),
        dpi=120,
        # optional per-call override:
        # show_projections=True,
        # projection_envelope=False,
    )
    frames.append(frame)

imageio.mimsave(out_dir / "embedding_orbit_loop.gif", frames, fps=50, loop=0)

In [ ]:
from xflow.utils import plot_image
plot_image(result[0][0])